# SALM Fine-tuning 05 · How it sounds, not what it says: speech emotion (CREMA-D)

> **Task** speech → one of 6 emotions · **Data** CREMA-D, split by actor · **Metric** accuracy ↑
> **Result** 36.71% → **76.67%** (+108.9%) · **Time** ~1.2 h

Part of the [SALM fine-tuning series](README.md); see [00_Overview_and_Setup](00_Overview_and_Setup.ipynb) first.

## The problem: an emotion detector that reads the words

In CREMA-D, 91 actors say the same twelve bland sentences ("It's eleven o'clock", "I think I have a doctor's
appointment") in six emotions: *anger, disgust, fear, happy, neutral, sad*. The words never change, so the **only**
evidence is *how* they're said: pitch, energy, tempo, voice quality.

A speech-to-text model listens to the words. Asked for the emotion, the base model mostly answers *neutral* or
*happy*, which is what the words sound like on paper. The goal is a model that recognizes emotion from the voice
alone, on actors it has never heard.

> Why not IEMOCAP, the classic? There the base model already scores well from the *words* alone (angry scripts are
> angry on paper), so a large gain would mean beating the state of the art. CREMA-D closes that shortcut.

In [ ]:
import json, os, shlex, subprocess, sys, time
from pathlib import Path


def find_nemo_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "examples" / "speechlm2" / "finetuning").is_dir():
            return candidate
    raise RuntimeError(f"Could not locate the NeMo source tree from {start}. Open this notebook from inside the repo checkout.")


NEMO_ROOT = find_nemo_root(Path.cwd())
FT = NEMO_ROOT / "examples" / "speechlm2" / "finetuning"              # the tools used below
WORK = Path(os.environ.get("SALM_FT_WORK", NEMO_ROOT / "tutorials" / "speechlm2" / "finetuning" / "workdir")).absolute()
os.environ["SALM_FT_WORK"] = str(WORK)                                # data, manifests, experiments, exports
os.environ.pop("MPLBACKEND", None)  # the kernel's inline plotting backend would break matplotlib in child processes
if not os.environ.get("SALM_CHECKPOINT"):
    raise RuntimeError(
        "SALM_CHECKPOINT is not set. Point it at your SALM checkpoint: a directory in HuggingFace format "
        "exported by SALMAutomodel. Run `export SALM_CHECKPOINT=/path/to/checkpoint` before "
        "starting Jupyter, or set os.environ['SALM_CHECKPOINT'] in a cell above this one."
    )
CKPT = Path(os.environ["SALM_CHECKPOINT"]).expanduser().absolute()  # base SALM checkpoint (HuggingFace format)
if not CKPT.is_dir():
    raise RuntimeError(f"SALM_CHECKPOINT={CKPT} is not a directory. It must be a SALM checkpoint in HuggingFace format.")
PYTHON = os.environ.get("TRAIN_PYTHON", "python")                     # NeMo training environment
VLLM_PYTHON = os.environ.get("VLLM_PYTHON", "python")                 # vLLM 0.28 environment (ideally a separate one)
M, EXP, EXPORT, EVAL = (WORK / d for d in ("manifests", "experiments", "exports", "eval"))
q = shlex.quote                                                        # shell-quote prompts (keeps non-ASCII intact)
for d in (M, EXP, EXPORT, EVAL, WORK / "logs", WORK / "configs"):
    d.mkdir(parents=True, exist_ok=True)


RETRIES = int(os.environ.get("SALM_FT_RETRIES", 3))                   # raise when other jobs share the GPU


def sh(cmd, log=None, retries=1):
    """Run a shell command from the NeMo root. With `log`, output goes to a file and only its tail is shown."""
    for attempt in range(1, retries + 1):
        print(f"$ {cmd}" + (f"   > {log}" if log else ""), flush=True)
        if log:
            with open(WORK / "logs" / log, "w") as f:
                rc = subprocess.run(cmd, shell=True, cwd=NEMO_ROOT, stdout=f, stderr=subprocess.STDOUT).returncode
            tail = (WORK / "logs" / log).read_text(errors="ignore").splitlines()
            print("\n".join(l for l in tail[-400:] if l.startswith(("[eval]", "[export]", "[scores]", "[merge]", "wrote")))[-3000:])
        else:
            rc = subprocess.run(cmd, shell=True, cwd=NEMO_ROOT).returncode
        if rc == 0:
            return
        if attempt < retries:  # e.g. a vLLM engine starting while another job allocates GPU memory
            print(f"(attempt {attempt} failed, retrying in 90 s)")
            time.sleep(90)
    raise RuntimeError(f"command failed (rc={rc}): {cmd}" + (f"; see {WORK / 'logs' / log}" if log else ""))


def evaluate(model, manifest, name, task_args, extra=""):
    """Decode `manifest` with vLLM (greedy) and return the metric summary.

    Re-running a cell reuses a previous result only if its fingerprint (model, manifest, prompt, decoder and scoring
    settings, code revision) matches; a result produced from different inputs is an error, never silently reused.
    """
    out = EVAL / f"{name}.jsonl"
    sh(f"{VLLM_PYTHON} {FT}/vllm_task_eval.py --model {model} --manifest {manifest} --out {out} {task_args} {extra} "
       f"--reuse", log=f"{name}.log", retries=RETRIES)
    return json.load(open(f"{out}.summary.json"))


def tidy(keep, name):
    """Free disk: delete this tutorial's other merged exports (~63 GB each) and leftover training checkpoints."""
    import shutil

    for p in EXPORT.glob(f"{name}_*"):
        if p != keep:
            shutil.rmtree(p)
    shutil.rmtree(EXP / name / "checkpoints", ignore_errors=True)
    print(f"kept {keep}; removed the other {name} exports and checkpoints")


def samples(name, n=3, keys=("text", "pred_text")):
    """Print a few decoded examples."""
    rows = [json.loads(l) for l in open(EVAL / f"{name}.jsonl")]
    for r in rows[:n]:
        print("  |  ".join(f"{k}: {str(r.get(k))[:110]}" for k in keys))


def result(name, value, expected, tol):
    """Compare a number with the expected result, and record it in results.json."""
    ok = abs(value - expected) <= tol
    print(f"{'✅' if ok else '❌'} {name}: {value:.4f}  (expected: {expected} ± {tol})")
    res = WORK / "results.json"
    d = json.loads(res.read_text()) if res.exists() else {}
    d[name] = {"value": value, "expected": expected, "tolerance": tol, "ok": ok}
    res.write_text(json.dumps(d, indent=1))
    return ok


print("NeMo root:", NEMO_ROOT, "\nwork dir: ", WORK, "\ncheckpoint:", CKPT)

## 1. Data: split by actor

The HF mirror's split is random over clips, so **every actor appears in every split**, and a model could learn *voices*
instead of emotions. The prep script re-splits by actor (seed 0): 68 train, 9 dev and 14 test actors.

In [ ]:
d = M / "cremad"
sh(f"{PYTHON} {FT}/prepare_benchmarks.py cremad", log="prep_cremad.log")
sh(f"{PYTHON} {FT}/prepare_benchmarks.py librispeech", log="prep_ls.log")
for s in ("train", "dev", "test"):
    rows = [json.loads(l) for l in open(d / f"{s}.json")]
    print(f"{s:5s} {len(rows):5d} clips from {len({r['actor'] for r in rows})} actors")

## 2. Baseline, and a check on the scorer

In [ ]:
TASK = f"--task cls --max-tokens 8 --labels {d}/labels.txt"
P2 = ("Ignore the words and listen to the tone of voice. Which emotion is the speaker acting: anger, disgust, fear, "
      "happy, neutral, or sad? Answer with one word.")
base = {}
for p, extra in (("p1", ""), ("p2", f"--prompt {q(P2)}")):
    for s in ("dev", "test"):
        base[p, s] = evaluate(CKPT, d / f"{s}.json", f"ser_base_{p}_{s}", TASK, extra)
    print(f"{p}  dev {100 * base[p, 'dev']['score']:5.2f}%   test {100 * base[p, 'test']['score']:5.2f}%")
BEST_P = max(("p1", "p2"), key=lambda p: base[p, "dev"]["score"])  # prompt chosen on dev
BASELINE = 100 * base[BEST_P, "test"]["score"]  # its test score: the baseline fine-tuning is compared with
ORACLE = 100 * max(base[p, "test"]["score"] for p in ("p1", "p2"))  # best prompt chosen on test: an upper bound
print(f"baseline prompt (chosen on dev): {BEST_P}; test-oracle (best prompt on test): {ORACLE:.2f}")
print(f"\nbaseline = {BASELINE:.2f}%")
print("base recall per emotion (p2, test):", {k: round(v, 2) for k, v in base["p2", "test"]["per_class_recall"].items()})

The prompt chosen on dev is p2 (36.71% on test, which is also the test-oracle). The base model leans on *neutral*
(99% recall) and *happy*, and almost never recognizes *disgust* or *sad* (5% recall each).

Check the raw outputs before trusting a baseline. The base model often answers **"angry"** instead of "anger". A
strict scorer counts those as invalid, which drops the baseline to ~30.9% and inflates the apparent improvement
(+148% instead of +109%). The scorer used here (`task_metrics.map_to_label`) maps synonyms onto the label set. When a
baseline "never" predicts a class, read the confusion matrix and the raw answers.

In [ ]:
pairs = " ".join(f"{d}/{s}.json {d}/{s}_ft.json" for s in ("train", "dev", "test"))
sh(f"{PYTHON} {FT}/set_context.py --template '{{context}}' {pairs}")   # train with p1: it lists the labels; training teaches the rest

## 3. Config: the encoder must learn

The label lives in the acoustics, so the speech encoder trains along with the adapters. 3.9 h of 2-second clips gives
~37 batches per epoch at 12k tokens.

In [ ]:
CFG = WORK / "configs" / "emotion.yaml"
sh(f"{PYTHON} {FT}/make_ft_config.py --checkpoint {CKPT} --prompt manifest --out {CFG} --exp-dir {EXP / 'emotion'} "
   f"--train-manifest {d}/train_ft.json --val-manifest {d}/dev_ft.json --val-name ser_dev "
   f"--train-encoder --train-proj --lr 1e-4 --warmup-steps 40 --max-steps 400 "
   f"--batch-tokens 12000 --limit-train-batches 30 --val-every-n-epochs 2 --limit-val-batches 30 --save-top-k -1")

## 4. Train (~2.3 s/step, ~20 min)

In [ ]:
sh(f"{PYTHON} {FT}/run_training.py --config {CFG} --preflight --smoke --keep-steps 120 360")

## 5. Select on dev

Expected dev accuracy: step 120 → 70.42%, **step 360 → 77.07%**. `val_loss` bottoms out at step 60 and rises from
there, yet dev accuracy keeps improving. Select by decoding, not by `val_loss`.

In [ ]:
dev = {}
for step in (120, 360):
    out = EXPORT / f"emotion_s{step}"
    sh(f"{PYTHON} {FT}/export_checkpoint.py --exp-dir {EXP / 'emotion'} --step {step} --base-checkpoint {CKPT} "
       f"--out {out} --delete-ckpt", log=f"export_emotion_{step}.log")
    dev[step] = 100 * evaluate(out, d / "dev_ft.json", f"emotion_s{step}_dev", TASK)["score"]
    print(f"step {step}: dev accuracy {dev[step]:.2f}%")
BEST = max(dev, key=dev.get)

## 6. Test on 14 unseen actors, once

In [ ]:
MODEL = EXPORT / f"emotion_s{BEST}"
ft = evaluate(MODEL, d / "test_ft.json", "emotion_TEST", TASK)
guard = evaluate(MODEL, M / "librispeech" / "test_clean.json", "emotion_librispeech", "--task asr --normalizer english")
FT_ACC = 100 * ft["score"]
print(f"test accuracy: base {BASELINE:.2f}% → fine-tuned {FT_ACC:.2f}% ({(FT_ACC - BASELINE) / BASELINE:+.1%}); LibriSpeech {100 * guard['score']:.2f}%")
print("recall per emotion:", {k: round(v, 2) for k, v in ft["per_class_recall"].items()})
result("05 baseline test accuracy", BASELINE, 36.71, 1.0)
result("05 fine-tuned test accuracy", FT_ACC, 76.67, 2.5)

Expected results (test accuracy, and recall per emotion in %):

| | test accuracy | anger | disgust | fear | happy | neutral | sad | LibriSpeech WER |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| base, prompt chosen on dev (p2) | 36.71% | 34 | 5 | 21 | 66 | 99 | 5 | 1.54% |
| **fine-tuned** | **76.67%** | 91 | 70 | 64 | 85 | 86 | 65 | 2.39% |

**+108.9% relative** on actors it never heard. Fear and sadness remain the hardest pair, as in the literature.

## Tidy up

Keep the selected model; delete the other exports (~63 GB each) and any leftover training checkpoints.

In [ ]:
tidy(keep=MODEL, name="emotion")

## Takeaways

1. **Pick a benchmark where the shortcut is closed.** CREMA-D forces the model to listen.
2. **Split by speaker.** Otherwise you measure voice memorization.
3. **Suspect the scorer before the model** when a baseline "never" does something.